#### Fuente: https://github.com/razekmh/World-University-Rankings-Table/blob/master/index.ipynb

#### 0. Setup de variables

In [1]:
import sys

sys.path.append("..")
from utils import SimpleLogger

logger = SimpleLogger(log_name="web_scraping")

In [2]:
from utils import Constantes

# Links para el Web Scraping
lk_wr = Constantes.lk_wr
lk_cs_scores = Constantes.lk_cs_scrs
lk_en_scores = Constantes.lk_en_scrs
lk_cs_stats = Constantes.lk_cs_stts
lk_en_stats = Constantes.lk_en_stts

# Columnas a extraer
scores_titles = Constantes.scores_titles
scores_ws_wr = Constantes.scores_ws_wr
scores_ws_su = Constantes.scores_ws_su
stats_titles = Constantes.stats_titles
stats_ws_wr = Constantes.stats_ws_wr
stats_ws_su = Constantes.stats_ws_su

print(f"lk_wr: {lk_wr}")
print(f"lk_cs_scores: {lk_cs_scores}")
print(f"lk_en_scores: {lk_en_scores}")
print(f"lk_cs_stats: {lk_cs_stats}")
print(f"lk_en_stats: {lk_en_stats}")

print(f"scores_titles: {scores_titles}")
print(f"scores_ws_wr: {scores_ws_wr}")
print(f"scores_ws_su: {scores_ws_su}")
print(f"stats_titles: {stats_titles}")
print(f"stats_ws_wr: {stats_ws_wr}")
print(f"stats_ws_su: {stats_ws_su}")

lk_wr: https://www.timeshighereducation.com/world-university-rankings/2025/world-ranking
lk_cs_scores: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/computer-science#!/length/-1/sort_by/rank/sort_order/asc/cols/scores
lk_en_scores: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/engineering-and-it#!/length/-1/sort_by/rank/sort_order/asc/cols/scores
lk_cs_stats: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/computer-science#!/length/-1/sort_by/rank/sort_order/asc/cols/stats
lk_en_stats: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/engineering-and-it#!/length/-1/sort_by/rank/sort_order/asc/cols/stats
scores_titles: ['rank', 'name', 'country', 'overall_score', 'research_qlt_score', 'industry_score', 'intl_outlook_score', 'research_env_score', 'teaching_score']
scores_ws_wr: ['cell-rank css-.*', 'cell-name css-.*', 'cell-scores_overall css-.

#### 1. Web scraping de 3 rankings (wr, cs, en) en 2 pestañas (ranking y key statistics)

##### Funciones de web scraping

In [3]:
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from tqdm import tqdm


def click_button(driver: WebDriver, btn_name: str) -> int:
    try:
        # Definir botón a presionar
        btn_str = f"//button[contains(., '{btn_name}')]"
        # Esperar a que el botón sea clicable
        btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, btn_str))
        )
        # Click al botón localizado
        btn.click()
        return 0
    except Exception as e:
        logger.exception(f"click_button: {e}")
        return


def web_scraper_wr(lk: str, change_tab: bool = False) -> list[BeautifulSoup]:
    # Valores constantes
    btn_reject = "Reject"
    btn_key_statistics = "Key statistics"
    # Cambia cada cierto tiempo
    # Maximiza en la pantalla pequeña e inspecciona el scroll
    table_scroll_id = "css-hzor4q"  # "css-gh9yfo"
    try:
        # Configurar proceso de web scraping
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        # Objeto tipo webdriver para el link
        driver = webdriver.Chrome(options=options)
        # Usar el webdriver para un request al link
        driver.get(lk)
        # Cerrar la ventana de cookies
        res = click_button(driver, btn_reject)
        if res is None:
            return
        # Scroll-down para visualizar tabla
        driver.execute_script("window.scrollBy(0, 800);")
        # Cambiar a ventana key_statistics
        if change_tab:
            time.sleep(2)
            res = click_button(driver, btn_key_statistics)
            if res is None:
                return
        # Parsear el HTML con BeautifulSoup de manera continua
        list_soup = []
        for _ in tqdm(range(20)):
            time.sleep(2)
            # Extrae el html
            soup = BeautifulSoup(driver.page_source, "html.parser")
            list_soup.append(soup)
            # Realiza un scroll-down
            scrollable_element = driver.find_element(By.CLASS_NAME, table_scroll_id)
            driver.execute_script("arguments[0].scrollTop += 6000", scrollable_element)
        driver.quit()
        return list_soup
    except Exception as e:
        logger.exception(f"web_scraper_wr: {e}")
        return


def web_scraper_su(lk: str) -> BeautifulSoup:
    # Valores constantes
    btn_reject = "Reject"
    try:
        # Configurar proceso de web scraping
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        # Objeto tipo webdriver para el link
        driver = webdriver.Chrome(options=options)
        # Usar el webdriver para un request al link
        driver.get(lk)
        # Cerrar la ventana de cookies
        res = click_button(driver, btn_reject)
        if res is None:
            return
        # Scroll-down para visualizar tabla
        driver.execute_script("window.scrollBy(0, 800);")
        # Parsear el HTML con BeautifulSoup una única vez
        soup = BeautifulSoup(driver.page_source, "html.parser")
        driver.quit()
        return soup
    except Exception as e:
        logger.exception(f"web_scraper_su: {e}")
        return

##### wr-scores

In [4]:
wr_scores = web_scraper_wr(lk_wr)

100%|██████████| 20/20 [02:07<00:00,  6.39s/it]


In [5]:
import re
import pandas as pd

df_wr_scores = pd.DataFrame()
for bloque in tqdm(wr_scores):
    reg_bloque = []
    for score in scores_ws_wr:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "cell-name css-.*":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="institution-link")
                if not aux2:
                    aux2 = fila.find("div", class_="css-vm0xc5")
                # pais
                aux3 = fila.find("span", class_="css-a7p034").text
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(scores_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_wr_scores.empty:
        df_wr_scores = df
    else:
        df_wr_scores = pd.concat([df_wr_scores, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_wr_scores.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 20/20 [00:10<00:00,  1.93it/s]


In [6]:
print(df_wr_scores.shape)
df_wr_scores.head()

(966, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,University of Oxford,United Kingdom,98.5,98.8,99.6,97.3,100.0,96.8
1,2,Massachusetts Institute of Technology,United States,98.1,99.7,100.0,93.8,96.0,99.2
2,3,Harvard University,United States,97.7,99.3,85.7,90.1,99.9,97.3
3,4,Princeton University,United States,97.5,98.9,96.9,87.4,98.0,98.3
4,5,University of Cambridge,United Kingdom,97.4,97.6,88.4,97.1,99.9,95.9


##### wr-stats

In [7]:
wr_stats = web_scraper_wr(lk_wr, True)

100%|██████████| 20/20 [02:06<00:00,  6.33s/it]


In [8]:
import re
import pandas as pd

df_wr_stats = pd.DataFrame()
for bloque in tqdm(wr_stats):
    reg_bloque = []
    for score in stats_ws_wr:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "cell-name css-.*":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="institution-link")
                if not aux2:
                    aux2 = fila.find("div", class_="css-vm0xc5")
                # pais
                aux3 = fila.find("span", class_="css-a7p034").text
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(stats_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_wr_stats.empty:
        df_wr_stats = df
    else:
        df_wr_stats = pd.concat([df_wr_stats, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_wr_stats.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 20/20 [00:07<00:00,  2.68it/s]


In [9]:
print(df_wr_stats.shape)
df_wr_stats.head()

(966, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,University of Oxford,United Kingdom,"22,095",10.8,43%,51 : 49
1,2,Massachusetts Institute of Technology,United States,"11,836",8.0,33%,42 : 58
2,3,Harvard University,United States,"22,584",10.0,25%,52 : 48
3,4,Princeton University,United States,"8,378",7.8,23%,47 : 53
4,5,University of Cambridge,United Kingdom,"20,980",11.5,38%,49 : 51


##### cs-scores

In [10]:
cs_scores = web_scraper_su(lk_cs_scores)

In [11]:
import re
import pandas as pd

df_cs_scores = pd.DataFrame()
for bloque in tqdm(cs_scores):
    reg_bloque = []
    for score in scores_ws_su:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "name namesearch":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="ranking-institution-title")
                if not aux2:
                    aux2 = fila.find("div", class_="ranking-institution-title")
                # pais
                aux3 = fila.find("a", href=re.compile(r"/location/"))
                if not aux3:
                    aux3 = fila.find(
                        "div", class_="ranking-institution__disabled-location"
                    )
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3.text)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(scores_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_cs_scores.empty:
        df_cs_scores = df
    else:
        df_cs_scores = pd.concat([df_cs_scores, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_cs_scores.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 1/1 [00:01<00:00,  1.09s/it]


In [12]:
print(df_cs_scores.shape)
df_cs_scores.head()

(1122, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,University of Oxford,United Kingdom,98.3,99.4,92.4,95.5,98.7,99.2
1,2,University of Cambridge,United Kingdom,97.3,95.6,99.5,96.1,100.0,95.8
2,3,Massachusetts Institute of Technology,United States,96.2,99.1,100.0,80.6,99.4,93.2
3,4,ETH Zurich,Switzerland,96.1,99.3,90.8,94.4,99.4,91.3
4,5,Stanford University,United States,96.0,99.8,100.0,76.7,98.8,93.5


##### cs_stats

In [13]:
cs_stats = web_scraper_su(lk_cs_stats)

In [14]:
import re
import pandas as pd

df_cs_stats = pd.DataFrame()
for bloque in tqdm(cs_stats):
    reg_bloque = []
    for score in stats_ws_su:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "name namesearch":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="ranking-institution-title")
                if not aux2:
                    aux2 = fila.find("div", class_="ranking-institution-title")
                # pais
                aux3 = fila.find("a", href=re.compile(r"/location/"))
                if not aux3:
                    aux3 = fila.find(
                        "div", class_="ranking-institution__disabled-location"
                    )
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3.text)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(stats_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_cs_stats.empty:
        df_cs_stats = df
    else:
        df_cs_stats = pd.concat([df_cs_stats, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_cs_stats.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


In [15]:
print(df_cs_stats.shape)
df_cs_stats.head()

(1122, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,University of Oxford,United Kingdom,"22,095",10.8,43%,51 : 49
1,2,University of Cambridge,United Kingdom,"20,980",11.5,38%,49 : 51
2,3,Massachusetts Institute of Technology,United States,"11,836",8.0,33%,42 : 58
3,4,ETH Zurich,Switzerland,"22,883",15.8,44%,33 : 67
4,5,Stanford University,United States,"16,963",5.9,23%,47 : 53


##### en-scores

In [16]:
en_scores = web_scraper_su(lk_en_scores)

In [17]:
import re
import pandas as pd

df_en_scores = pd.DataFrame()
for bloque in tqdm(en_scores):
    reg_bloque = []
    for score in scores_ws_su:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "name namesearch":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="ranking-institution-title")
                if not aux2:
                    aux2 = fila.find("div", class_="ranking-institution-title")
                # pais
                aux3 = fila.find("a", href=re.compile(r"/location/"))
                if not aux3:
                    aux3 = fila.find(
                        "div", class_="ranking-institution__disabled-location"
                    )
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3.text)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(scores_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_en_scores.empty:
        df_en_scores = df
    else:
        df_en_scores = pd.concat([df_en_scores, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_en_scores.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


In [18]:
print(df_en_scores.shape)
df_en_scores.head()

(1488, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,Harvard University,United States,97.5,97.6,97.1,93.7,99.6,96.5
1,2,Stanford University,United States,96.4,98.6,100.0,86.4,95.3,97.0
2,3,Massachusetts Institute of Technology,United States,96.2,97.9,100.0,87.9,93.0,99.0
3,4,University of Oxford,United Kingdom,96.1,92.9,95.7,95.1,99.8,95.7
4,5,"University of California, Berkeley",United States,95.8,98.8,100.0,86.5,98.5,91.2


##### en-stats

In [19]:
en_stats = web_scraper_su(lk_en_stats)

In [20]:
import re
import pandas as pd

df_en_stats = pd.DataFrame()
for bloque in tqdm(en_stats):
    reg_bloque = []
    for score in stats_ws_su:
        aux = bloque.findAll("td", {"class": re.compile(score)})
        if score == "name namesearch":
            reg_score_1 = []
            reg_score_2 = []
            for fila in aux:
                # universidad
                aux2 = fila.find("a", class_="ranking-institution-title")
                if not aux2:
                    aux2 = fila.find("div", class_="ranking-institution-title")
                # pais
                aux3 = fila.find("a", href=re.compile(r"/location/"))
                if not aux3:
                    aux3 = fila.find(
                        "div", class_="ranking-institution__disabled-location"
                    )
                reg_score_1.append(aux2.text)
                reg_score_2.append(aux3.text)
            reg_bloque.append(reg_score_1)
            reg_bloque.append(reg_score_2)
        else:
            reg_score = [fila.text for fila in aux]
            reg_bloque.append(reg_score)
    # Se ha construido una lista con 9 listas con la misma cantidad de
    # componentes
    data_dict = dict(zip(stats_titles, reg_bloque))
    df = pd.DataFrame(data_dict)
    # Concatenear distintos bloques
    if df_en_stats.empty:
        df_en_stats = df
    else:
        df_en_stats = pd.concat([df_en_stats, df], ignore_index=True)
# Quitar duplicados de bloques distintos
df_en_stats.drop_duplicates(subset=["name", "country"], keep="first", inplace=True)

100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


In [21]:
print(df_en_stats.shape)
df_en_stats.head()

(1488, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,Harvard University,United States,"22,584",10.0,25%,52 : 48
1,2,Stanford University,United States,"16,963",5.9,23%,47 : 53
2,3,Massachusetts Institute of Technology,United States,"11,836",8.0,33%,42 : 58
3,4,University of Oxford,United Kingdom,"22,095",10.8,43%,51 : 49
4,5,"University of California, Berkeley",United States,"42,423",18.7,25%,52 : 48


##### Guardar rankings en csv

In [22]:
from utils import Constantes
import os

ruta_csv = f"{Constantes.ruta_reports}/raw"
os.makedirs(ruta_csv, exist_ok=True)

df_wr_scores.to_csv(f"{ruta_csv}/wr_scores.csv", index=False)
df_wr_stats.to_csv(f"{ruta_csv}/wr_stats.csv", index=False)
df_cs_scores.to_csv(f"{ruta_csv}/cs_scores.csv", index=False)
df_cs_stats.to_csv(f"{ruta_csv}/cs_stats.csv", index=False)
df_en_scores.to_csv(f"{ruta_csv}/en_scores.csv", index=False)
df_en_stats.to_csv(f"{ruta_csv}/en_stats.csv", index=False)